In [9]:
import os
# Importar bibliotecas necesarias para memoria
from langchain_openai import ChatOpenAI
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
import gradio as gr

# Configurar cliente OpenAI
try:
    llm = ChatOpenAI(
        base_url=os.getenv("OPENAI_BASE_URL"),
        api_key=os.getenv("GITHUB_TOKEN"),
        model="gpt-4o-mini",
        streaming=True,
        temperature=0.5
    )

    print("✓ Modelo configurado para experimentos de memoria")
    print(f"Modelo: {llm.model_name}")

except Exception as e: 
    print(f"✗ Error en configuración: {e}")
    print("Verifica las variables de entorno")

# Almacén de historiales
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


# Crear conversación
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un experto en todos las arias de TCG ya sea de pokemon, yugioh, riftbound, etc., que controla sobre valor de sobres y cartas individuales. Respondes con información precisa y actualizada sobre el valor de cartas y sobres, así como consejos para coleccionistas y jugadores."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

conversation = RunnableWithMessageHistory(
    prompt | llm,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

def chat_interactivo():
    print("=== CHAT INTERACTIVO CON MEMORIA ===")
    print("Escribe 'salir' para terminar la conversación\n")

    session_id = "chat_session"

    while True:
        try:
            # Pedir input del usuario
            user_input = input("👤 Tú: ").strip()

            if user_input.lower() in ['salir', 'exit', 'quit']:
                print("¡Hasta luego!")
                break

            if not user_input:
                continue

            # Mostrar respuesta con streaming
            print("🤖 Asistente: ", end="", flush=True)
            
            full_response = ""
            
            # Usar stream() para obtener respuestas en tiempo real
            for chunk in conversation.stream(
                {"input": user_input},
                config={"configurable": {"session_id": session_id}}
            ):
                # Extraer el contenido del chunk
                if hasattr(chunk, 'content'):
                    content = chunk.content
                else:
                    content = str(chunk)
                
                print(content, end="", flush=True)
                full_response += content
            
            print("\n")

        except KeyboardInterrupt:
            print("\n¡Conversación interrumpida!")
            break
        except Exception as e:
            print(f"Error: {e}")
            break

# Ejecutar chat interactivo
if __name__ == "__main__":
    
    chat_interactivo()

✗ Error en configuración: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable
Verifica las variables de entorno


NameError: name 'llm' is not defined

***Interfaz con GRADIO***

In [ ]:
import os
from openai import OpenAI
# Importar bibliotecas necesarias para memoria
from langchain_openai import ChatOpenAI
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
import gradio as gr

# Configurar cliente OpenAI
try:
    llm = ChatOpenAI(
        base_url=os.getenv("OPENAI_BASE_URL"),
        api_key=os.getenv("GITHUB_TOKEN"),
        model="gpt-4o-mini",
        streaming=True,
        temperature=0.5
    )

    print("✓ Modelo configurado para experimentos de memoria")
    print(f"Modelo: {llm.model_name}")

except Exception as e:
    print(f"✗ Error en configuración: {e}")
    print("Verifica las variables de entorno")

# Almacén de historiales
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


# Crear conversación
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un experto en todos las arias de TCG ya sea de pokemon, yugioh, riftbound, etc., que controla sobre valor de sobres y cartas individuales. Respondes con información precisa y actualizada sobre el valor de cartas y sobres, así como consejos para coleccionistas y jugadores."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

conversation = RunnableWithMessageHistory(
    prompt | llm,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

def respond(message, history):
    response = conversation.invoke(
        {"input": message}, 
        config={"configurable": {"session_id": "gradio_session"}}
    )
    return response.content

# Crear interfaz Gradio
demo = gr.ChatInterface(
    respond,
    title="Asistente TCG Experto",
    description="Pregunta sobre valores de cartas y sobres de TCG como Pokémon, Yu-Gi-Oh, etc."
)

# Lanzar la interfaz
demo.launch()

✗ Error en configuración: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable
Verifica las variables de entorno


NameError: name 'llm' is not defined

## Tecnicas IL1.2 integradas al chatbot TCG

| Tecnica | Donde se aplica |
|---|---|
| **Zero-Shot** | System prompt con rol, tarea, formato y restricciones claras |
| **Few-Shot** | Ejemplos de consultas TCG reales dentro del prompt |
| **Chain-of-Thought** | Instruccion de razonamiento paso a paso para analisis de precios |

In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
import gradio as gr

# =============================================================
# TECNICAS IL1.2 INTEGRADAS
# Zero-Shot: system prompt con rol, tarea, formato, restricciones
# Few-Shot:  ejemplos de consultas TCG reales en el prompt
# Chain-of-Thought: instruccion de razonamiento paso a paso
# =============================================================

FULL_SYSTEM = "Eres un experto en Trading Card Games (TCG) con amplio conocimiento en\nPokemon, Yu-Gi-Oh!, Magic: The Gathering, Riftbound y otros juegos de cartas.\n\nTarea: Responder consultas sobre precios de cartas y sobres, ayudar a coleccionistas\ny jugadores con informacion precisa sobre el mercado TCG.\n\nFormato de respuesta:\n- Precio aproximado en USD/EUR con rango (minimo - maximo)\n- Factores que afectan el valor (edicion, estado, rareza)\n- Consejo practico para el usuario\n\nRestricciones:\n- Si no tienes datos actualizados del precio, indicalo claramente\n- No inventes precios; da rangos realistas o recomienda consultar TCGPlayer/Cardmarket\n- Responde en el idioma del usuario\n\n# --- FEW-SHOT: Ejemplos de consultas y respuestas TCG ---\nEjemplos de como responder:\n\nConsulta: 'Cuanto vale un Charizard de base set?'\nRespuesta: El Charizard Holo de la Base Set (1999) tiene un valor aproximado de:\n- PSA 10: $300,000+ USD (ejemplares perfectos son raros)\n- PSA 9: $10,000 - $30,000 USD\n- Sin grading / buen estado: $500 - $2,000 USD\nFactores clave: el estado es fundamental; una esquina doblada puede reducir el valor a la mitad.\nConsejo: si vas a vender, considera gradear con PSA o Beckett primero.\n\nConsulta: 'Que tan caro es abrir sobres de Phantom Nightmare de Yu-Gi-Oh?'\nRespuesta: Un sobre de Phantom Nightmare (2024) cuesta aproximadamente $4-5 USD por sobre.\n- Caja de 24 sobres: $90-110 USD\n- Cartas mas valiosas del set: Floowandereeze & Empen (~$15), Yubel Terror Incarnate (~$30)\nFactores clave: la EV (Expected Value) de una caja suele estar por debajo del costo, abrir es por experiencia.\nConsejo: si buscas las cartas especificas, comprelas sueltas en Cardmarket es mas economico.\n\n# --- CHAIN-OF-THOUGHT: Razonamiento paso a paso para analisis de precios ---\nCuando analices el valor de una carta o producto TCG, razona internamente:\n1. Identifica el juego, carta/producto y edicion mencionados\n2. Considera los factores que afectan el precio (rareza, estado, demanda)\n3. Estima el rango de precio segun esos factores\n4. Da una recomendacion concreta al usuario\n"

# Modelo con temperatura baja para mayor consistencia en precios
llm = ChatOpenAI(
    base_url=os.getenv('OPENAI_BASE_URL'),
    api_key=os.getenv('GITHUB_TOKEN'),
    model='gpt-4o-mini',
    streaming=True,
    temperature=0.3
)

store_v2 = {}

def get_session_history_v2(session_id: str):
    if session_id not in store_v2:
        store_v2[session_id] = InMemoryChatMessageHistory()
    return store_v2[session_id]

prompt_v2 = ChatPromptTemplate.from_messages([
    ('system', FULL_SYSTEM),
    MessagesPlaceholder(variable_name='chat_history'),
    ('human', '{input}')
])

conversation_v2 = RunnableWithMessageHistory(
    prompt_v2 | llm,
    get_session_history_v2,
    input_messages_key='input',
    history_messages_key='chat_history'
)

def respond_tcg(message, history):
    response = conversation_v2.invoke(
        {'input': message},
        config={'configurable': {'session_id': 'gradio_v2'}}
    )
    return response.content

demo_v2 = gr.ChatInterface(
    respond_tcg,
    title='Asistente TCG Experto - IL1.2',
    description=(
        'Consulta precios de cartas y sobres TCG.\n'
        'Tecnicas aplicadas: Zero-Shot + Few-Shot + Chain-of-Thought'
    ),
    examples=[
        'Cuanto vale un Charizard de base set en buen estado?',
        'Vale la pena abrir sobres de la ultima expansion de Pokemon?',
        'Cual es la carta mas cara de Yu-Gi-Oh actualmente?',
        'Como saber si una carta de Magic es valiosa?'
    ]
    
)

demo_v2.launch()


OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable